# 05 · Train with PyTorch Lightning — Multi Node

Multi-node 토폴로지(M×N). 단일 노드는 [`04-train_pytorch_lightning_single_node.ipynb`](04-train_pytorch_lightning_single_node.ipynb).

데이터셋·모델·하이퍼파라미터는 1×1·1×N과 동일(`CONFIG`, ML-25M). 차이는 `num_nodes=M`과 worker 클러스터뿐.

Multi-node에서는 `ddp_notebook`이 동작하지 않으므로 TorchDistributor가 worker process를 띄우고, 각 process 안에서 `Trainer(strategy='ddp')`가 이미 세팅된 `RANK`/`WORLD_SIZE`/`MASTER_ADDR` 환경변수를 그대로 사용합니다.

**클러스터**: Multi-node Classic GPU, DBR 17.3 LTS ML. Driver `g5.12xlarge` + Workers `g5.12xlarge` × M. **Autoscaling OFF 필수**. Single node 토글 OFF. Serverless GPU 사용 불가 ([databricks-environments.md](../00-foundations/databricks-environments.md)).

TorchDistributor child 안에 모든 import·클래스 정의를 다시 둡니다 ([common-pitfalls.md §2](../00-foundations/common-pitfalls.md)).

In [ ]:
%run ./00-setup

## 학습 함수

TorchDistributor child가 실행할 entrypoint. LightningModule/DataModule을 함수 내부에 정의하고, `MLFlowLogger(run_id=...)`로 driver의 run에 attach합니다.

In [ ]:
def train_fn_lightning_mxn(
    experiment_path,
    run_id,
    db_host,
    db_token,
    data_dir,
    ckpt_dir,
    n_users,
    n_items,
    emb_dim,
    tower_hidden,
    batch_size,
    num_epochs,
    max_steps_per_epoch,
    patience,
    min_delta,
    num_gpus_per_node,
    num_nodes,
):
    import os

    import lightning as L
    import mlflow
    import pyarrow.parquet as pq
    import torch
    import torch.nn as nn
    from lightning.pytorch.callbacks import EarlyStopping
    from lightning.pytorch.loggers import MLFlowLogger
    from torch.utils.data import DataLoader, TensorDataset

    os.environ["DATABRICKS_HOST"] = db_host
    os.environ["DATABRICKS_TOKEN"] = db_token

    class TwoTowerLitModule(L.LightningModule):
        def __init__(self, n_users, n_items, emb_dim, tower_hidden, lr=1e-3, weight_decay=1e-5):
            super().__init__()
            self.save_hyperparameters()
            self.user_emb = nn.Embedding(n_users, emb_dim)
            self.item_emb = nn.Embedding(n_items, emb_dim)
            self.user_tower = self._make_tower(emb_dim, tower_hidden)
            self.item_tower = self._make_tower(emb_dim, tower_hidden)
            self.loss_fn = nn.BCEWithLogitsLoss()

        @staticmethod
        def _make_tower(in_dim, hidden):
            layers, prev = [], in_dim
            for h in hidden:
                layers += [nn.Linear(prev, h), nn.ReLU()]
                prev = h
            return nn.Sequential(*layers)

        def forward(self, user_ids, item_ids):
            u = self.user_tower(self.user_emb(user_ids))
            i = self.item_tower(self.item_emb(item_ids))
            return (u * i).sum(dim=-1)

        def training_step(self, batch, batch_idx):
            u, i, y = batch
            logits = self(u, i)
            loss = self.loss_fn(logits, y)
            self.log("train/loss", loss, on_step=True, sync_dist=True)
            return loss

        def validation_step(self, batch, batch_idx):
            u, i, y = batch
            logits = self(u, i)
            loss = self.loss_fn(logits, y)
            self.log("val/loss", loss, on_step=False, on_epoch=True, sync_dist=True)
            return loss

        def configure_optimizers(self):
            return torch.optim.AdamW(
                self.parameters(), lr=self.hparams.lr, weight_decay=self.hparams.weight_decay
            )

    class InteractionsDataModule(L.LightningDataModule):
        def __init__(self, data_dir, batch_size, num_workers=2):
            super().__init__()
            self.data_dir = data_dir
            self.batch_size = batch_size
            self.num_workers = num_workers

        def _load_split(self, split):
            split_dir = os.path.join(self.data_dir, split)
            files = sorted(
                os.path.join(split_dir, f) for f in os.listdir(split_dir) if f.endswith(".parquet")
            )
            table = pq.read_table(files, columns=["user_id", "item_id", "label"])
            return TensorDataset(
                torch.from_numpy(table.column("user_id").to_numpy()),
                torch.from_numpy(table.column("item_id").to_numpy()),
                torch.from_numpy(table.column("label").to_numpy()),
            )

        def setup(self, stage=None):
            self.train_dataset = self._load_split("train")
            self.val_dataset = self._load_split("val")

        def train_dataloader(self):
            return DataLoader(
                self.train_dataset,
                batch_size=self.batch_size,
                shuffle=True,
                num_workers=self.num_workers,
                pin_memory=True,
            )

        def val_dataloader(self):
            return DataLoader(
                self.val_dataset,
                batch_size=self.batch_size,
                shuffle=False,
                num_workers=self.num_workers,
                pin_memory=True,
            )

    model = TwoTowerLitModule(
        n_users=n_users, n_items=n_items, emb_dim=emb_dim, tower_hidden=tower_hidden
    )
    dm = InteractionsDataModule(data_dir=data_dir, batch_size=batch_size)

    logger = MLFlowLogger(
        experiment_name=experiment_path,
        tracking_uri="databricks",
        run_id=run_id,
    )
    early_stop = EarlyStopping(
        monitor="val/loss",
        patience=patience,
        min_delta=min_delta,
        mode="min",
    )
    trainer = L.Trainer(
        accelerator="gpu",
        devices=num_gpus_per_node,
        num_nodes=num_nodes,
        strategy="ddp",
        max_epochs=num_epochs,
        limit_train_batches=max_steps_per_epoch,
        logger=logger,
        callbacks=[early_stop],
        default_root_dir=ckpt_dir,
    )

    rank = int(os.environ.get("RANK", "0"))
    if rank == 0:
        # worker 노드(rank 0)에서 driver의 run에 attach해 system metrics를 수집한다.
        # driver는 학습에 참여하지 않으므로 driver-side log_system_metrics만으로는 idle 메트릭만 잡힌다.
        mlflow.start_run(run_id=run_id, log_system_metrics=True)
    try:
        trainer.fit(model, datamodule=dm)
    finally:
        if rank == 0 and mlflow.active_run() is not None:
            mlflow.end_run()
    return "ok"

## M×N GPU

`NUM_NODES`, `NUM_GPUS_PER_NODE`를 클러스터 구성에 맞춥니다. 1×1·1×N과 같은 `CONFIG`·같은 `DATA_DIR` 사용.

In [ ]:
import os

import mlflow
from pyspark.ml.torch.distributor import TorchDistributor

NUM_NODES = 2          # M
NUM_GPUS_PER_NODE = 4  # N

cfg = CONFIG
mlflow.set_experiment(EXPERIMENT_PATH)
with mlflow.start_run(run_name="lightning-MxN", log_system_metrics=True) as run:
    run_id = run.info.run_id

TorchDistributor(
    num_processes=NUM_NODES * NUM_GPUS_PER_NODE,
    local_mode=False,
    use_gpu=True,
).run(
    train_fn_lightning_mxn,
    experiment_path=EXPERIMENT_PATH,
    run_id=run_id,
    db_host=DB_HOST,
    db_token=DB_TOKEN,
    data_dir=DATA_DIR,
    ckpt_dir=os.path.join(CKPT_DIR, "lightning-MxN"),
    n_users=cfg["n_users"],
    n_items=cfg["n_items"],
    emb_dim=cfg["emb_dim"],
    tower_hidden=cfg["tower_hidden"],
    batch_size=cfg["batch_size_per_gpu"],
    num_epochs=cfg["num_epochs"],
    max_steps_per_epoch=cfg["max_steps_per_epoch"],
    patience=PATIENCE,
    min_delta=MIN_DELTA,
    num_gpus_per_node=NUM_GPUS_PER_NODE,
    num_nodes=NUM_NODES,
)